In [1]:
from config import MISSING_THRESHOLD
from utils.print_section import print_section

"""
04_dataset_selection.ipynb
"""

'\n04_dataset_selection.ipynb\n'

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from utils.load_data_cleaned import load_data_cleaned

df = load_data_cleaned()

In [3]:
import pandas as pd
from utils.print_section import print_section

# -------------------------------------------------------------------
# Keep BV/BVBA
# -------------------------------------------------------------------
df = df[
    df["rechtsvorm"].isin(["BV", "BVBA"])
]

# ============================================================
# Dataset size
# ============================================================
print_section("Basic shape")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ============================================================
# Number of companies
# ============================================================
print_section("Unique companies")

print(
    f"Unique VAT numbers: "
    f"{df['vat'].nunique():,}"
)

# ============================================================
# Panel structure
# ============================================================
print_section("Panel structure")

obs_per_company = df.groupby("vat").size()

print(obs_per_company.describe())

# ============================================================
# Bookyears
# ============================================================
print_section("Bookyears")

print(
    df["bookyear"]
    .value_counts()
    .sort_index()
)

# ============================================================
# Legal forms
# ============================================================
print_section("Legal forms")

print(
    df["rechtsvorm"]
    .value_counts(dropna=False)
    .sort_index()
)

# ============================================================
# Accounting schemas
# ============================================================
print_section("Schema types")

print(
    df["nature"]
    .value_counts(dropna=False)
    .sort_index()
)

# ============================================================
# Industries
# ============================================================
print_section("Industries")

print(
    df["industry"]
    .value_counts(dropna=False)
    .sort_index()
)

# ============================================================
# Bankruptcies
# ============================================================
print_section("Bankruptcies")

failed_companies = (
    df.loc[df["jaar_van_faling"].notna(), "vat"]
    .nunique()
)

total_companies = df["vat"].nunique()

fail_rate = (
    failed_companies / total_companies * 100
)

print(f"Unique failed companies: {failed_companies:,}")
print(f"Failure rate: {fail_rate:.2f}%")

# ============================================================
# Bankruptcy years
# ============================================================
print_section("Bankruptcy years")

print(
    df["jaar_van_faling"]
    .value_counts(dropna=False)
    .sort_index()
)


# ============================================================
# t0 -> t1 analysis
# ============================================================
print_section("T0 -> T1 availability")

results = []

for year in sorted(df["bookyear"].unique()):

    n_obs = (
        df["bookyear"] == year
    ).sum()

    n_fail = (
        (
            (df["bookyear"] == year)
            &
            (df["jaar_van_faling"] == year + 1)
        )
        .sum()
    )

    results.append(
        {
            "bookyear": year,
            "observations": n_obs,
            "fail_next_year": n_fail,
        }
    )

summary = pd.DataFrame(results)

summary["fail_rate_pct"] = (
    summary["fail_next_year"]
    / summary["observations"]
    * 100
).round(3)

print(summary)

df = df[
    df["nature"].isin([1, 2, 7])
]

df = df[
    df["bookyear"].isin([2017, 2018, 2019])
]

df["target"] = (
    df["jaar_van_faling"]
    == df["bookyear"] + 1
).astype(int)


# ============================================================
# Target distribution
# ============================================================
print_section("Target distribution")

print(
    df["target"]
    .value_counts()
)

print()

print(
    (
        df["target"]
        .value_counts(normalize=True)
        * 100
    ).round(3)
)

# ============================================================
# Removing missing values
# ============================================================

print_section("Remove high-missing columns after sample selection")

PROTECTED_COLUMNS = [
    "jaar_van_faling",
    "fail",
    "faling_datum",
    "target",
]

missing_pct = df.isna().mean()

drop_cols = [
    col
    for col in missing_pct[
        missing_pct >= 0.90
    ].index
    if col not in PROTECTED_COLUMNS
]

print(f"Columns to remove: {len(drop_cols)}")

for col in sorted(drop_cols):
    print(col)

# Save shape before dropping columns
original_rows = len(df)
original_columns = len(df.columns)
original_companies = df["vat"].nunique()

df = df.drop(columns=drop_cols)

print(
    f"\nDataset shape after column removal: "
    f"{df.shape}"
)

# ============================================================
# Save model dataset
# ============================================================

OUTPUT_PATH = "../data/processed/model_dataset.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Model dataset saved to: {OUTPUT_PATH}")

# ============================================================
# Write log
# ============================================================

print_section("Write drop log")

LOG_PATH = "../logs/model_dataset_log.txt"

target_counts = df["target"].value_counts()
target_pct = (
    df["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

with open(LOG_PATH, "w") as f:

    f.write("MODEL DATASET CONSTRUCTION LOG\n")
    f.write("=" * 60 + "\n\n")

    f.write("Selection criteria\n")
    f.write("-" * 60 + "\n")

    f.write("rechtsvorm: BV, BVBA\n")
    f.write("nature: 1, 2, 7\n")
    f.write("bookyear: 2017, 2018, 2019\n")
    f.write("target: jaar_van_faling == bookyear + 1\n\n")

    f.write(
        f"Original dataset: "
        f"{original_rows:,} rows x "
        f"{original_columns:,} columns\n"
    )

    f.write(
        f"Final dataset: "
        f"{len(df):,} rows x "
        f"{len(df.columns):,} columns\n"
    )

    f.write(
        f"Unique companies: "
        f"{original_companies:,}\n\n"
    )

    f.write("Target distribution\n")
    f.write("-" * 60 + "\n")

    for target_value in sorted(target_counts.index):

        count = target_counts[target_value]
        pct = target_pct[target_value]

        f.write(
            f"{target_value}: "
            f"{count:,} "
            f"({pct:.3f}%)\n"
        )

    f.write("\n")

    f.write(
        f"High-missing columns removed "
        f"({len(drop_cols)})\n"
    )

    f.write("-" * 60 + "\n")

    for col in sorted(drop_cols):
        f.write(f"{col}\n")

print(f"Log written to: {LOG_PATH}")


Basic shape
Rows: 2,550,039
Columns: 106

Unique companies
Unique VAT numbers: 373,680

Panel structure
count    373680.000000
mean          6.824125
std           2.839130
min           1.000000
25%           5.000000
50%           9.000000
75%           9.000000
max           9.000000
dtype: float64

Bookyears
bookyear
2010     22092
2011    203387
2012    228592
2013    245907
2014    266684
2015    285973
2016    303456
2017    320353
2018    336357
2019    325365
2020     11873
Name: count, dtype: int64

Legal forms
rechtsvorm
BV       162008
BVBA    2388031
Name: count, dtype: int64

Schema types
nature
1     1721472
2       36411
3           9
4          15
5           2
7      791759
81        157
82         21
87        193
Name: count, dtype: int64

Industries
industry
0.0      61900
10.0     38762
11.0     47145
12.0     47488
13.0     29007
14.0    886132
15.0    192715
16.0    488786
17.0    403189
18.0    273830
19.0     81085
Name: count, dtype: int64

Bankruptcies
Uniq